In [1]:
!pip install -U "spacy[transformers]"
!pip install -U tokenizers
!pip install -U transformers
!pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -U "la-core-web-trf @ https://huggingface.co/latincy/la_core_web_trf/resolve/main/la_core_web_trf-any-py3-none-any.whl"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.8/758.8 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [11]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
python -m spacy train config.cfg \
  --output ./output \
  --paths.train ./corpus/train.spacy \
  --paths.dev ./corpus/val.spacy \
  --code ./functions.py \
  --gpu-id 0

ℹ Saving to output directory:
/content/drive/MyDrive/master_thesis/training/model/spacy/output
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
2025-07-16 01:49:20.842846: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752630560.864026   21963 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752630560.872479   21963 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Some weights of RobertaModel were not initialized from the model checkpoint at pstroe/roberta-base-latin-cased and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-st

In [13]:
!python -m spacy evaluate drive/MyDrive/master_thesis/training/model/spacy/output/model-best \
  drive/MyDrive/master_thesis/training/model/spacy/corpus/test.spacy \
  --code drive/MyDrive/master_thesis/training/model/spacy/la_core_web_lg/functions.py \
  --output drive/MyDrive/master_thesis/training/model/spacy/test_eval.json

ℹ Using CPU
ℹ To switch to GPU 0, use the option: --gpu-id 0
2025-07-16 02:29:31.334102: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752632971.631273   32318 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752632971.714538   32318 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered

================================== Results ==================================

TOK     100.00
NER P   75.25 
NER R   72.82 
NER F   74.01 
SPEED   26    


=============================== NER (per type) ===============================

                 P       R       F
PERS:PRAE    85.48   88.33   86.89
PERS:NOMEN   83.58   76.02   79.62
PERS:TITLE   80.60   90.00   85.04

In [14]:
import spacy
from spacy.tokens import DocBin
from pathlib import Path
from sklearn.metrics import classification_report
import importlib.util
import sys
from pathlib import Path

code_path = Path("drive/MyDrive/master_thesis/training/model/spacy/la_core_web_lg/functions.py")
spec = importlib.util.spec_from_file_location("functions", code_path)
module = importlib.util.module_from_spec(spec)
sys.modules["functions"] = module
spec.loader.exec_module(module)

model_path = "drive/MyDrive/master_thesis/training/model/spacy/output/model-best"
test_path = "drive/MyDrive/master_thesis/training/model/spacy/corpus/test.spacy"

nlp = spacy.load(model_path)
doc_bin = DocBin().from_disk(test_path)
docs = list(doc_bin.get_docs(nlp.vocab))

true_labels = []
pred_labels = []

for doc in docs:
    pred_doc = nlp(doc.text)

    for token in doc:
        if token.is_space:
            continue

        gold_tag = "O"
        for ent in doc.ents:
            if token.idx >= ent.start_char and token.idx < ent.end_char:
                prefix = "B" if token.idx == ent.start_char else "I"
                gold_tag = f"{prefix}-{ent.label_}"
                break
        true_labels.append(gold_tag)

        pred_tag = "O"
        for ent in pred_doc.ents:
            if token.idx >= ent.start_char and token.idx < ent.end_char:
                prefix = "B" if token.idx == ent.start_char else "I"
                pred_tag = f"{prefix}-{ent.label_}"
                break
        pred_labels.append(pred_tag)

print("Model Evaluation Report:")
print(classification_report(true_labels, pred_labels, digits=4))

Model Evaluation Report:
              precision    recall  f1-score   support

       B-LOC     0.2778    0.2632    0.2703        19
      B-PERS     0.5875    0.5402    0.5629        87
   B-PERS:AG     0.5763    0.6296    0.6018        54
  B-PERS:COG     0.7788    0.7535    0.7660       215
 B-PERS:FILI     0.8551    0.8082    0.8310        73
B-PERS:NOMEN     0.8358    0.7602    0.7962       221
 B-PERS:PRAE     0.8548    0.8833    0.8689       120
B-PERS:TITLE     0.8060    0.9000    0.8504        60
     B-TITLE     0.7465    0.6795    0.7114        78
       I-LOC     0.6667    0.3636    0.4706        11
      I-PERS     0.0000    0.0000    0.0000         5
   I-PERS:AG     0.3000    0.3000    0.3000        10
 I-PERS:FILI     0.8889    0.8713    0.8800       101
I-PERS:TITLE     0.0000    0.0000    0.0000         1
     I-TITLE     0.8421    0.6154    0.7111        52
           O     0.9481    0.9729    0.9604      2105

    accuracy                         0.8913      3212
 

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
